# 04 — Human Dose Estimation

Calculate Human Equivalent Dose (HED) and Maximum Recommended Starting Dose (MRSD) from animal data.

**FDA BSA Method:**
```
HED (mg/kg) = Animal Dose × (Animal Km / Human Km)
MRSD = HED / Safety Factor (default: 10)
```

In [ ]:
import sys
sys.path.insert(0, '..')

from doseprojection.human_dose import calc_hed, calc_mrsd, hed_from_noael, bsa_conversion
from doseprojection.constants import KM_FACTORS
import pandas as pd

## HED from Rat NOAEL

In [ ]:
result = hed_from_noael(noael_mg_kg=50, species="rat", safety_factor=10)

print("=== HED/MRSD from Rat NOAEL ===")
print(f"Rat NOAEL: {result['noael_mg_kg']} mg/kg")
print(f"HED: {result['hed_mg_kg']:.2f} mg/kg")
print(f"HED (total for 60 kg human): {result['hed_total_mg']:.0f} mg")
print(f"MRSD: {result['mrsd_mg_kg']:.3f} mg/kg")
print(f"MRSD (total): {result['mrsd_total_mg']:.1f} mg")

## Compare HED Across Species

Same NOAEL measured in different species — the most sensitive species (lowest HED) drives the MRSD.

In [ ]:
noaels = {
    'mouse': 100,
    'rat': 50,
    'dog': 10,
    'monkey': 15,
}

rows = []
for species, noael in noaels.items():
    r = hed_from_noael(noael, species)
    rows.append({
        'Species': species,
        'NOAEL (mg/kg)': noael,
        'Km': KM_FACTORS[species],
        'HED (mg/kg)': round(r['hed_mg_kg'], 2),
        'HED (total mg)': round(r['hed_total_mg'], 0),
        'MRSD (mg/kg)': round(r['mrsd_mg_kg'], 3),
        'MRSD (total mg)': round(r['mrsd_total_mg'], 1),
    })

df = pd.DataFrame(rows)
print("Select the species yielding the LOWEST HED as the most sensitive species.")
df

## Interspecies Dose Conversion

Convert doses between any two species using BSA normalization.

In [ ]:
# Convert 30 mg/kg mouse dose to rat equivalent
result = bsa_conversion(30, 'mouse', 'rat')
print(f"Mouse 30 mg/kg → Rat: {result['dose_mg_kg']:.1f} mg/kg")

# Convert 10 mg/kg dog dose to monkey equivalent
result = bsa_conversion(10, 'dog', 'monkey')
print(f"Dog 10 mg/kg → Monkey: {result['dose_mg_kg']:.1f} mg/kg")